# GPU Memory Subsystem: Registers, Shared Memory/L1, L2, and HBM

In modern deep learning systems—especially during LLM decoding—memory bandwidth, not compute, is almost always the primary bottleneck.

1. The GPU Memory Hierarchy at a Glance
The memory hierarchy is governed by a fundamental hardware law: The faster the memory, the smaller its capacity and the closer it must physically sit to the ALUs.

┌────────────────────────────────────────────────────────────────────────────────┐
│ REGISTERS (Fastest, Smallest)                                                  │
│ ~256 KB per SM | Latency: ~0-1 clock cycles | Bandwidth: ~30-40 TB/s aggregate │
└──────────────────────────────────────┬─────────────────────────────────────────┘
                                       │
┌──────────────────────────────────────▼─────────────────────────────────────────┐
│ SHARED MEMORY / L1 DATA CACHE (On-Chip SRAM)                                   │
│ ~128 KB - 228 KB per SM | Latency: ~20-30 cycles | Bandwidth: ~15-20 TB/s      │
└──────────────────────────────────────┬─────────────────────────────────────────┘
                                       │
┌──────────────────────────────────────▼─────────────────────────────────────────┐
│ L2 CACHE (Die-Level Shared SRAM)                                               │
│ ~40 MB - 50 MB per GPU | Latency: ~150-200 cycles | Bandwidth: ~5-7 TB/s       │
└──────────────────────────────────────┬─────────────────────────────────────────┘
                                       │
┌──────────────────────────────────────▼─────────────────────────────────────────┐
│ HIGH-BANDWIDTH MEMORY / HBM (Off-Chip DRAM / "VRAM")                           │
│ 80 GB - 144 GB | Latency: ~400-800 cycles | Bandwidth: 2.0 - 3.35 TB/s         │
└──────────────────────────────────────┬─────────────────────────────────────────┘
                                       │
┌──────────────────────────────────────▼─────────────────────────────────────────┐
│ HOST SYSTEM MEMORY (CPU RAM via PCIe Gen5 / NVLink C2C)                        │
│ 512 GB - 2 TB | Latency: Thousands of cycles | Bandwidth: ~64 - 128 GB/s (PCIe)│
└────────────────────────────────────────────────────────────────────────────────┘

The Quantitative Reality (Latency & Bandwidth Comparison)Memory LevelLocationScope / AccessibilityTypical Size (e.g., H100)Latency (Cycles)Aggregated BandwidthRegistersOn-Chip (SM)Private to 1 Thread$256\text{ KB}$ per SM ($\sim 33\text{ MB}$ total)$\sim 1$$\sim 30+\text{ TB/s}$Shared Memory / L1On-Chip (SM)Shared across 1 BlockUp to $228\text{ KB}$ per SM ($\sim 30\text{ MB}$ total)$\sim 20 - 30$$\sim 15 - 20\text{ TB/s}$L2 CacheOn-Die (Shared)Accessible by All SMs$50\text{ MB}$ total$\sim 150 - 200$$\sim 6\text{ TB/s}$Global Memory (HBM3)Off-Die on SubstrateAccessible by Entire Grid & Host$80\text{ GB}$$\sim 400 - 800$$\sim 3.35\text{ TB/s}$Host RAM (PCIe)Host MotherboardCPU DRAM$512\text{ GB}+$$> 2000+$$\sim 64\text{ GB/s}$ (PCIe 5.0 $\times 16$)Notice the cliff:Moving from Registers $\rightarrow$ HBM is a $400\times$ to $800\times$ latency penalty.Moving from HBM $\rightarrow$ CPU RAM over PCIe is another $30\times$ bandwidth drop.